# Agent-to-Agent (A2A) Protocol — Standardized Agent Communication

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/30_a2a_protocol.ipynb)

## What This Notebook Teaches You

In the previous notebook, we built MCP to connect agents to **tools**. But what happens when one agent needs help from *another agent*? Today, agents are isolated — a research agent can't ask a writing agent for help without custom glue code. **Agent-to-Agent (A2A)** is Google's protocol for solving this.

**Research foundation**:
- Google, *"Agent2Agent (A2A) Protocol Specification"* (2025) — an open protocol for inter-agent communication
- The core insight: agents need to **discover**, **communicate with**, and **delegate to** other agents, just like microservices discover and call each other

By the end of this notebook, you will:

1. **Understand why agents need a communication protocol** — isolation, discovery, delegation
2. **Build Agent Cards** — capability advertisements that let agents describe what they can do
3. **Implement an Agent Directory** — a registry where agents publish and discover capabilities
4. **Design a Task Protocol** — structured requests, responses, and status tracking
5. **Build A2A Agents** that discover and delegate tasks to each other
6. **Run a 3-agent collaborative system** — research, analysis, and writing agents working together
7. **Compare MCP vs A2A** — tool access vs. agent collaboration

---

> **Prerequisites**: Notebooks 01–06, Notebook 29 (MCP).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## Part 0: Environment Setup

Same Qwen3-8B setup. The model will power multiple agents that communicate through the A2A protocol — each agent gets its own system prompt and capabilities.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. What Is A2A?

### 1.1 — The Problem: Agent Isolation

Today's AI agents are islands:

```
┌──────────┐     ┌──────────┐     ┌──────────┐
│ Research  │     │ Analysis │     │ Writing  │
│  Agent    │     │  Agent   │     │  Agent   │
│          │     │          │     │          │
│ (alone)  │     │ (alone)  │     │ (alone)  │
└──────────┘     └──────────┘     └──────────┘
     ↑                ↑                ↑
  Custom API      Custom API      Custom API
     ↑                ↑                ↑
  App 1            App 2            App 3
```

Each agent:
- **Can't discover** other agents that might help
- **Can't delegate** subtasks to specialists
- **Can't compose** with other agents dynamically
- Uses a **custom integration** for every interaction

### 1.2 — The Solution: A Protocol for Agent Communication

A2A standardizes how agents find and talk to each other:

```
┌──────────┐     ┌──────────┐     ┌──────────┐
│ Research  │◄───►│ Analysis │◄───►│ Writing  │
│  Agent    │     │  Agent   │     │  Agent   │
│          │ A2A │          │ A2A │          │
│ card: ..│     │ card: ..│     │ card: ..│
└──────────┘     └──────────┘     └──────────┘
       ▲              ▲               ▲
       └──────────────┼───────────────┘
                      │
              ┌───────────────┐
              │ Agent         │
              │ Directory     │
              │ (discovery)   │
              └───────────────┘
```

### 1.3 — Key Concepts

1. **Agent Card**: A JSON document describing an agent's capabilities (like a business card)
2. **Agent Directory**: A registry where agents publish cards and discover other agents
3. **Task Protocol**: Structured format for requesting, tracking, and completing work
4. **Dynamic Discovery**: Agents find collaborators at runtime based on capability matching

## 2. Research Context: MCP and A2A — Two Complementary Protocols

These two protocols solve different problems:

| | **MCP** (Anthropic, 2024) | **A2A** (Google, 2025) |
|---|---|---|
| **Connects** | Agents to **tools** | Agents to **agents** |
| **Analogy** | USB — plug in a device | TCP/IP — computers talk to each other |
| **Discovery** | "What tools do you have?" | "What can you do? Can you help me?" |
| **Interaction** | Request → execute → result | Task request → collaborate → deliver |
| **Relationship** | Client-server (agent uses tool) | Peer-to-peer (agents collaborate) |

**They work together**: An agent might use MCP to access a database tool, then use A2A to delegate analysis of the results to a specialist agent.

```
User → Agent A  ──MCP──►  Database Server (get data)
                ──A2A──►  Agent B (analyze data)
                ──A2A──►  Agent C (write report)
```

## 3. Agent Cards — Capability Advertisement

The foundation of A2A is the **Agent Card**. Every agent publishes a card that describes:
- **Who** it is (name, version)
- **What** it can do (list of capabilities/skills)
- **How** to reach it (endpoint)
- **What** authentication it requires

Think of it like a microservice's OpenAPI specification, but for an AI agent's *cognitive capabilities* rather than API endpoints.

In [ ]:
# ============================================================
# Agent Card — capability advertisement
# ============================================================

@dataclass
class AgentCard:
    """Describes an agent's identity and capabilities.

    In the real A2A spec, this is published at /.well-known/agent.json
    so other agents can discover it via HTTP.
    """
    name: str                          # human-readable agent name
    description: str                   # what this agent does
    capabilities: List[str]            # list of skills/capabilities
    endpoint: str = ""                 # how to reach this agent
    version: str = "1.0"              # agent version
    auth_required: bool = False        # whether auth is needed
    max_concurrent_tasks: int = 5      # how many tasks it can handle
    supported_input_types: List[str] = field(default_factory=lambda: ["text"])
    supported_output_types: List[str] = field(default_factory=lambda: ["text"])

    def to_dict(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "capabilities": self.capabilities,
            "endpoint": self.endpoint,
            "version": self.version,
            "authRequired": self.auth_required,
            "maxConcurrentTasks": self.max_concurrent_tasks,
            "supportedInputTypes": self.supported_input_types,
            "supportedOutputTypes": self.supported_output_types,
        }

    def matches_capability(self, query: str) -> float:
        """Score how well this agent matches a capability query (0.0 to 1.0).

        Uses keyword overlap — a real implementation would use embeddings.
        """
        query_words = set(query.lower().split())
        card_words = set()
        for cap in self.capabilities:
            card_words.update(cap.lower().split())
        card_words.update(self.description.lower().split())

        if not query_words:
            return 0.0
        overlap = query_words & card_words
        return len(overlap) / len(query_words)

# Create example agent cards
research_card = AgentCard(
    name="ResearchAgent",
    description="Searches for information and gathers relevant data on any topic",
    capabilities=["research", "search", "information retrieval", "fact finding", "data gathering"],
    endpoint="agent://research-agent:8001"
)

analysis_card = AgentCard(
    name="AnalysisAgent",
    description="Analyzes data, compares options, and provides structured evaluations",
    capabilities=["analysis", "comparison", "evaluation", "reasoning", "data interpretation"],
    endpoint="agent://analysis-agent:8002"
)

writing_card = AgentCard(
    name="WritingAgent",
    description="Writes, formats, and summarizes content in clear prose",
    capabilities=["writing", "formatting", "summarization", "editing", "content creation"],
    endpoint="agent://writing-agent:8003"
)

print("=== Agent Cards ===\n")
for card in [research_card, analysis_card, writing_card]:
    print(f"{card.name} (v{card.version})")
    print(f"  Description: {card.description}")
    print(f"  Capabilities: {', '.join(card.capabilities)}")
    print(f"  Endpoint: {card.endpoint}")
    print()

## 4. Agent Directory — Discovery Service

The Agent Directory is a registry where agents publish their cards and discover other agents. It answers the question: *"Who can help me with X?"*

This is analogous to:
- **DNS** for the internet (name → address)
- **Service registry** in microservices (Consul, etcd)
- **App Store** for mobile apps (search by capability)

Without a directory, agents would need hardcoded knowledge of every other agent — exactly the fragmentation problem that killed early web services.

In [ ]:
# ============================================================
# Agent Directory — agent discovery service
# ============================================================

class AgentDirectory:
    """Registry where agents publish capabilities and discover each other."""

    def __init__(self):
        self.agents: Dict[str, AgentCard] = {}  # name → card
        self._log: List[str] = []

    def register(self, card: AgentCard) -> bool:
        """Register an agent's card in the directory."""
        if card.name in self.agents:
            self._log.append(f"Updated: {card.name}")
        else:
            self._log.append(f"Registered: {card.name}")
        self.agents[card.name] = card
        return True

    def unregister(self, name: str) -> bool:
        """Remove an agent from the directory."""
        if name in self.agents:
            del self.agents[name]
            self._log.append(f"Unregistered: {name}")
            return True
        return False

    def discover(self, capability_query: str, min_score: float = 0.1) -> List[Tuple[AgentCard, float]]:
        """Find agents matching a capability query, ranked by relevance."""
        matches = []
        for card in self.agents.values():
            score = card.matches_capability(capability_query)
            if score >= min_score:
                matches.append((card, score))
        matches.sort(key=lambda x: x[1], reverse=True)
        return matches

    def list_all(self) -> List[AgentCard]:
        """List all registered agents."""
        return list(self.agents.values())

    def get(self, name: str) -> Optional[AgentCard]:
        """Get a specific agent's card."""
        return self.agents.get(name)

    def get_log(self) -> List[str]:
        return list(self._log)

# Create directory and register agents
directory = AgentDirectory()
for card in [research_card, analysis_card, writing_card]:
    directory.register(card)

print("=== Agent Directory ===")
print(f"Registered agents: {len(directory.list_all())}\n")

# Test discovery with different queries
queries = [
    "search for information",
    "analyze and compare data",
    "write a summary",
    "evaluate research findings",  # should match both analysis and research
]

for q in queries:
    matches = directory.discover(q)
    print(f'Query: "{q}"')
    for card, score in matches:
        print(f"  → {card.name} (score: {score:.2f})")
    print()

## 5. Task Protocol — Structured Work Requests

In A2A, agents communicate through **tasks** — structured work requests that include:
- What needs to be done (description + input data)
- Who's asking (sender identity)
- Priority and metadata
- Status tracking for long-running tasks

This is more than just sending a message — it's a complete **task lifecycle**:

```
SUBMITTED → IN_PROGRESS → COMPLETED
                       ↘ FAILED
                       ↘ DELEGATED (handed to another agent)
```

In [ ]:
# ============================================================
# Task Protocol — structured task requests and responses
# ============================================================

import uuid
from enum import Enum

class TaskStatus(Enum):
    SUBMITTED = "submitted"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    DELEGATED = "delegated"

@dataclass
class TaskRequest:
    """A structured work request from one agent to another."""
    task_id: str
    sender: str                    # name of requesting agent
    description: str               # what needs to be done
    input_data: Dict[str, Any]     # task-specific input
    priority: int = 1              # 1 (low) to 5 (urgent)
    context: str = ""              # additional context
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "taskId": self.task_id,
            "sender": self.sender,
            "description": self.description,
            "inputData": self.input_data,
            "priority": self.priority,
            "context": self.context,
            "metadata": self.metadata,
        }

@dataclass
class TaskResponse:
    """A response to a task request."""
    task_id: str
    status: TaskStatus
    result: Any = None             # the actual output
    error_message: str = ""        # explanation if failed
    delegated_to: str = ""         # name of agent if delegated
    processing_time: float = 0.0   # seconds taken

    def to_dict(self) -> dict:
        d = {
            "taskId": self.task_id,
            "status": self.status.value,
            "result": self.result,
            "processingTime": self.processing_time,
        }
        if self.error_message:
            d["errorMessage"] = self.error_message
        if self.delegated_to:
            d["delegatedTo"] = self.delegated_to
        return d

@dataclass
class TaskRecord:
    """Complete record of a task's lifecycle."""
    request: TaskRequest
    response: Optional[TaskResponse] = None
    status: TaskStatus = TaskStatus.SUBMITTED
    created_at: float = field(default_factory=time.time)
    updated_at: float = field(default_factory=time.time)

    def update_status(self, status: TaskStatus):
        self.status = status
        self.updated_at = time.time()

# Demonstrate task creation
task = TaskRequest(
    task_id=str(uuid.uuid4())[:8],
    sender="UserAgent",
    description="Research the benefits of renewable energy",
    input_data={"topic": "renewable energy", "depth": "comprehensive"},
    priority=3,
    context="Preparing a report for stakeholders"
)

print("=== Task Request ===")
print(json.dumps(task.to_dict(), indent=2))

response = TaskResponse(
    task_id=task.task_id,
    status=TaskStatus.COMPLETED,
    result="Renewable energy reduces carbon emissions by 80% compared to fossil fuels...",
    processing_time=2.5
)
print("\n=== Task Response ===")
print(json.dumps(response.to_dict(), indent=2))

## 6. Building the A2A Agent

Each A2A agent is a self-contained entity that:
1. **Has an identity** (Agent Card) describing what it can do
2. **Registers** with an Agent Directory so others can find it
3. **Receives tasks** from other agents and processes them
4. **Discovers** other agents when it needs help
5. **Delegates** subtasks to specialists

The key difference from a regular agent: an A2A agent is both a **client** (sends tasks) and a **server** (receives tasks).

In [ ]:
# ============================================================
# A2A Agent — can discover, send, and receive tasks
# ============================================================

class A2AAgent:
    """An agent that communicates with other agents via the A2A protocol."""

    def __init__(self, card: AgentCard, directory: AgentDirectory,
                 system_prompt: str = "You are a helpful assistant."):
        self.card = card
        self.directory = directory
        self.system_prompt = system_prompt

        # Task tracking
        self.task_inbox: List[TaskRecord] = []    # tasks received
        self.task_outbox: List[TaskRecord] = []   # tasks sent
        self._log: List[str] = []

        # Register with directory
        self.directory.register(self.card)
        self._log.append(f"Agent '{self.card.name}' registered with directory")

    def receive_task(self, request: TaskRequest) -> TaskResponse:
        """Process an incoming task request."""
        record = TaskRecord(request=request)
        self.task_inbox.append(record)
        self._log.append(f"Received task {request.task_id} from {request.sender}: {request.description[:50]}...")

        record.update_status(TaskStatus.IN_PROGRESS)
        start_time = time.time()

        try:
            result = self._process_task(request)
            elapsed = time.time() - start_time
            response = TaskResponse(
                task_id=request.task_id,
                status=TaskStatus.COMPLETED,
                result=result,
                processing_time=elapsed
            )
            record.response = response
            record.update_status(TaskStatus.COMPLETED)
            self._log.append(f"Completed task {request.task_id} in {elapsed:.1f}s")
            return response

        except Exception as e:
            elapsed = time.time() - start_time
            response = TaskResponse(
                task_id=request.task_id,
                status=TaskStatus.FAILED,
                error_message=str(e),
                processing_time=elapsed
            )
            record.response = response
            record.update_status(TaskStatus.FAILED)
            self._log.append(f"Failed task {request.task_id}: {e}")
            return response

    def _process_task(self, request: TaskRequest) -> str:
        """Use the LLM to process a task. Override for custom logic."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": self._format_task_prompt(request)}
        ]
        return generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)

    def _format_task_prompt(self, request: TaskRequest) -> str:
        """Convert a task request into an LLM prompt."""
        parts = [f"Task: {request.description}"]
        if request.context:
            parts.append(f"Context: {request.context}")
        if request.input_data:
            for key, val in request.input_data.items():
                parts.append(f"{key}: {val}")
        return "\n".join(parts)

    def send_task(self, target_name: str, description: str,
                  input_data: Dict[str, Any] = None, priority: int = 1,
                  context: str = "") -> TaskResponse:
        """Send a task to another agent by name."""
        target_card = self.directory.get(target_name)
        if not target_card:
            return TaskResponse(
                task_id="none",
                status=TaskStatus.FAILED,
                error_message=f"Agent '{target_name}' not found in directory"
            )

        request = TaskRequest(
            task_id=str(uuid.uuid4())[:8],
            sender=self.card.name,
            description=description,
            input_data=input_data or {},
            priority=priority,
            context=context
        )

        record = TaskRecord(request=request)
        self.task_outbox.append(record)
        self._log.append(f"Sent task {request.task_id} to {target_name}: {description[:50]}...")

        # In a real A2A implementation, this would be an HTTP call to target_card.endpoint
        # Here we call the target agent directly (simulating network communication)
        target_agent = _agent_registry.get(target_name)
        if target_agent is None:
            return TaskResponse(
                task_id=request.task_id,
                status=TaskStatus.FAILED,
                error_message=f"Agent '{target_name}' not reachable"
            )

        response = target_agent.receive_task(request)
        record.response = response
        record.update_status(response.status)
        return response

    def discover_agents(self, capability: str) -> List[Tuple[str, float]]:
        """Discover agents that match a capability query."""
        matches = self.directory.discover(capability)
        # Don't return self
        results = [(card.name, score) for card, score in matches if card.name != self.card.name]
        self._log.append(f"Discovered {len(results)} agents for '{capability}'")
        return results

    def get_log(self) -> List[str]:
        return list(self._log)

    def get_stats(self) -> Dict[str, Any]:
        return {
            "name": self.card.name,
            "tasks_received": len(self.task_inbox),
            "tasks_sent": len(self.task_outbox),
            "completed": sum(1 for r in self.task_inbox if r.status == TaskStatus.COMPLETED),
            "failed": sum(1 for r in self.task_inbox if r.status == TaskStatus.FAILED),
        }

# Global agent registry (simulates network reachability)
_agent_registry: Dict[str, A2AAgent] = {}

print("A2AAgent class defined ✓")
print("Agents can discover, send, and receive tasks via A2A protocol.")

## 7. Example: 3-Agent Collaborative System

Let's build a system with three specialist agents:

```
User Request: "Analyze the pros and cons of remote work"
      │
      ▼
┌────────────┐    "Research remote work   ┌────────────┐
│  Research   │───  statistics and       ──►│  Analysis   │
│  Agent      │    findings"               │  Agent      │
│             │                            │             │
│ Skills:     │    "Analyze and compare   │ Skills:     │
│ - research  │     these findings"        │ - analysis  │
│ - search    │                            │ - compare   │
└────────────┘                            └──────┬─────┘
                                                  │
                "Write a clear summary"           │
                                                  ▼
                                           ┌────────────┐
                                           │  Writing    │
                                           │  Agent      │
                                           │             │
                                           │ Skills:     │
                                           │ - writing   │
                                           │ - summarize │
                                           └────────────┘
```

Each agent processes its specialty and can delegate to the next.

In [ ]:
# ============================================================
# 3-Agent Collaborative System
# ============================================================

# Clear global registry
_agent_registry.clear()

# Create specialized agents with domain-specific system prompts
research_agent = A2AAgent(
    card=research_card,
    directory=directory,
    system_prompt="""You are a Research Agent. Your specialty is gathering information.
When given a research task, provide relevant facts, statistics, and key findings.
Be thorough but concise. Present information as structured bullet points.
Focus on factual content that other agents can analyze further."""
)

analysis_agent = A2AAgent(
    card=analysis_card,
    directory=directory,
    system_prompt="""You are an Analysis Agent. Your specialty is evaluating information.
When given data or findings, analyze patterns, compare perspectives, identify strengths
and weaknesses, and provide structured evaluation. Use clear categories like
'Advantages', 'Disadvantages', 'Key Tradeoffs'. Be balanced and evidence-based."""
)

writing_agent = A2AAgent(
    card=writing_card,
    directory=directory,
    system_prompt="""You are a Writing Agent. Your specialty is clear, polished communication.
When given analysis or findings, write them up in clear, well-structured prose.
Use headers, topic sentences, and logical flow. Make the content accessible
to a general audience while preserving accuracy."""
)

# Register in global registry (simulates network)
_agent_registry["ResearchAgent"] = research_agent
_agent_registry["AnalysisAgent"] = analysis_agent
_agent_registry["WritingAgent"] = writing_agent

print("=== 3-Agent System Ready ===\n")
for name, agent in _agent_registry.items():
    print(f"  {name}: {agent.card.description}")
print(f"\nDirectory has {len(directory.list_all())} agents registered")

In [ ]:
# ============================================================
# Run the 3-agent pipeline: Research → Analysis → Writing
# ============================================================

topic = "benefits and challenges of remote work"
print(f"=== Multi-Agent Pipeline ===")
print(f"Topic: {topic}\n")

# Step 1: Research Agent gathers information
print("--- Step 1: Research Agent ---")
research_response = research_agent.receive_task(TaskRequest(
    task_id=str(uuid.uuid4())[:8],
    sender="User",
    description=f"Research the {topic}. Provide key facts, statistics, and findings.",
    input_data={"topic": topic, "depth": "comprehensive"},
    priority=3
))
print(f"Status: {research_response.status.value}")
print(f"Time: {research_response.processing_time:.1f}s")
print(f"Result preview: {str(research_response.result)[:300]}...\n")

# Step 2: Analysis Agent evaluates the research
print("--- Step 2: Analysis Agent ---")
analysis_response = analysis_agent.receive_task(TaskRequest(
    task_id=str(uuid.uuid4())[:8],
    sender="ResearchAgent",
    description="Analyze these research findings. Compare pros and cons, identify key tradeoffs.",
    input_data={"findings": str(research_response.result)[:500]},
    priority=3,
    context=f"Original topic: {topic}"
))
print(f"Status: {analysis_response.status.value}")
print(f"Time: {analysis_response.processing_time:.1f}s")
print(f"Result preview: {str(analysis_response.result)[:300]}...\n")

# Step 3: Writing Agent creates the final output
print("--- Step 3: Writing Agent ---")
writing_response = writing_agent.receive_task(TaskRequest(
    task_id=str(uuid.uuid4())[:8],
    sender="AnalysisAgent",
    description="Write a clear, well-structured summary based on this analysis.",
    input_data={"analysis": str(analysis_response.result)[:500]},
    priority=3,
    context=f"Original topic: {topic}"
))
print(f"Status: {writing_response.status.value}")
print(f"Time: {writing_response.processing_time:.1f}s")
print(f"\n=== Final Output ===\n{writing_response.result}")

## 8. Agent-to-Agent Delegation

In the previous example, we orchestrated the pipeline manually. The real power of A2A is when agents **discover and delegate to each other autonomously**.

Let's build a coordinator agent that:
1. Receives a complex task
2. Discovers specialist agents
3. Delegates subtasks
4. Assembles the final result

In [ ]:
# ============================================================
# Coordinator Agent — orchestrates other agents via A2A
# ============================================================

class CoordinatorAgent(A2AAgent):
    """An agent that coordinates other agents to complete complex tasks."""

    def _process_task(self, request: TaskRequest) -> str:
        """Break down a task and delegate to specialists."""
        self._log.append(f"Coordinating task: {request.description[:50]}...")

        # Step 1: Plan the subtasks
        plan = self._plan_subtasks(request)
        self._log.append(f"Plan: {len(plan)} subtasks")

        # Step 2: Execute each subtask (discover + delegate)
        results = {}
        for subtask in plan:
            capability = subtask["capability"]
            description = subtask["description"]

            # Discover best agent for this capability
            candidates = self.discover_agents(capability)
            if not candidates:
                results[capability] = f"No agent found for '{capability}'"
                continue

            best_agent_name, score = candidates[0]
            self._log.append(f"Delegating '{capability}' to {best_agent_name} (score: {score:.2f})")

            # Send task to the best match
            response = self.send_task(
                target_name=best_agent_name,
                description=description,
                input_data=subtask.get("input_data", {}),
                context=request.description
            )

            if response.status == TaskStatus.COMPLETED:
                results[capability] = str(response.result)
            else:
                results[capability] = f"Failed: {response.error_message}"

        # Step 3: Synthesize results
        return self._synthesize(request, results)

    def _plan_subtasks(self, request: TaskRequest) -> List[Dict]:
        """Use LLM to break down a task into subtasks."""
        return [
            {
                "capability": "research search information",
                "description": f"Research key facts about: {request.description}",
                "input_data": request.input_data
            },
            {
                "capability": "analysis comparison evaluation",
                "description": f"Analyze and evaluate findings about: {request.description}",
                "input_data": request.input_data
            },
            {
                "capability": "writing summarization formatting",
                "description": f"Write a clear summary about: {request.description}",
                "input_data": request.input_data
            }
        ]

    def _synthesize(self, request: TaskRequest, results: Dict[str, str]) -> str:
        """Combine results from multiple agents into a final response."""
        parts = [f"=== Coordinated Result for: {request.description} ===\n"]
        for capability, result in results.items():
            parts.append(f"--- {capability.title()} Phase ---")
            parts.append(str(result)[:400])
            parts.append("")
        return "\n".join(parts)

# Create coordinator
coordinator_card = AgentCard(
    name="CoordinatorAgent",
    description="Orchestrates complex tasks by discovering and delegating to specialist agents",
    capabilities=["coordination", "orchestration", "task management", "delegation"],
    endpoint="agent://coordinator:8000"
)

coordinator = CoordinatorAgent(
    card=coordinator_card,
    directory=directory,
    system_prompt="You are a coordinator that breaks down tasks and delegates to specialists."
)
_agent_registry["CoordinatorAgent"] = coordinator

print("=== Coordinator Agent ===")
print(f"Registered: {coordinator.card.name}")
print(f"Directory now has {len(directory.list_all())} agents")
print(f"\nThe coordinator will discover and delegate to specialist agents automatically.")

In [ ]:
# ============================================================
# Coordinator auto-delegates to specialist agents
# ============================================================

task = TaskRequest(
    task_id=str(uuid.uuid4())[:8],
    sender="User",
    description="Explain the impact of artificial intelligence on healthcare",
    input_data={"topic": "AI in healthcare", "audience": "general public"},
    priority=4
)

print(f"User sends task to Coordinator: {task.description}\n")
print("The coordinator will:")
print("  1. Break the task into subtasks")
print("  2. Discover specialist agents for each")
print("  3. Delegate and collect results")
print("  4. Synthesize a final answer\n")
print("=" * 60)

response = coordinator.receive_task(task)

print(f"\nStatus: {response.status.value}")
print(f"Processing time: {response.processing_time:.1f}s")
print(f"\n{response.result}")

print("\n--- Coordinator Log ---")
for entry in coordinator.get_log():
    print(f"  {entry}")

## 9. Dynamic Discovery — Hot-Plugging New Agents

Just like MCP servers can be added dynamically, A2A agents can **register at any time** and become immediately discoverable by all other agents.

This is how agent ecosystems scale: new specialists join and the system's capabilities grow — no restarts, no code changes, no central updates.

In [ ]:
# ============================================================
# Dynamic agent discovery — add a new specialist at runtime
# ============================================================

# Show current capabilities
print("=== Before: Available Agents ===")
for card in directory.list_all():
    print(f"  {card.name}: {', '.join(card.capabilities)}")

# New agent joins the system
translation_card = AgentCard(
    name="TranslationAgent",
    description="Translates text between languages with cultural awareness",
    capabilities=["translation", "language", "localization", "multilingual"],
    endpoint="agent://translation-agent:8004"
)

translation_agent = A2AAgent(
    card=translation_card,
    directory=directory,
    system_prompt="""You are a Translation Agent. Translate text accurately while
preserving tone and cultural nuances. When asked, provide the translation
along with notes about any cultural adaptations made."""
)
_agent_registry["TranslationAgent"] = translation_agent

print(f"\n>>> New agent registered: {translation_card.name}")
print(f"\n=== After: Available Agents ===")
for card in directory.list_all():
    print(f"  {card.name}: {', '.join(card.capabilities)}")

# Existing agents can immediately discover the new one
print("\n=== Discovery Test ===")
matches = research_agent.discover_agents("translate to another language")
print(f'Research agent searches for "translate to another language":')
for name, score in matches:
    print(f"  → {name} (score: {score:.2f})")

# The coordinator can now delegate translation tasks
print("\n=== Coordinator discovers translation capability ===")
matches = coordinator.discover_agents("translation language")
for name, score in matches:
    print(f"  → {name} (score: {score:.2f})")
print("\nThe coordinator can now delegate translation subtasks automatically!")

## 10. Comparison: Direct Message-Passing vs. A2A Protocol

### Direct Message-Passing (Ad Hoc)
```python
# Every agent-to-agent interaction is custom
result = agent_b.process(agent_a.output)  # tight coupling
# Adding Agent C? Modify Agent A and B's code
# Different message format? Rewrite everything
```

### A2A Protocol (Standardized)
```python
# Standard discovery + structured tasks
candidates = directory.discover("analysis")
response = agent.send_task(candidates[0], task_request)
# Adding Agent C? Just register it — zero changes to A or B
```

| Aspect | Direct Messaging | A2A Protocol |
|--------|-----------------|--------------|
| **Coupling** | Tight — agents know each other | Loose — agents discover at runtime |
| **Adding agents** | Modify existing code | Just register new agent |
| **Error handling** | Custom per pair | Standard TaskResponse status |
| **Discoverability** | Hardcoded | Agent Directory with capability matching |
| **Task tracking** | Manual | Built-in TaskRecord lifecycle |
| **Scalability** | O(n²) integrations | O(n) registrations |
| **Interoperability** | None | Any A2A-compliant agent works |

## 11. MCP vs A2A — When to Use Each

These protocols solve fundamentally different problems and are **complementary**, not competing:

### MCP: Agent ↔ Tool Communication
```
Agent  ──MCP──►  Tool Server
"Call the calculator"     "Here's the result"
```
- **Relationship**: Client (agent) → Server (tools)
- **Use when**: An agent needs to access external tools, APIs, databases
- **Analogy**: Using a screwdriver (tool) to build furniture

### A2A: Agent ↔ Agent Communication
```
Agent A  ──A2A──►  Agent B
"Can you analyze this?"   "Here's my analysis"
```
- **Relationship**: Peer-to-peer (both are agents)
- **Use when**: A task requires capabilities from multiple specialist agents
- **Analogy**: Asking a colleague (another expert) for help

### Using Them Together
```
User → Coordinator Agent
              │
              ├──A2A──► Research Agent ──MCP──► Web Search Server
              │                        ──MCP──► Database Server
              │
              ├──A2A──► Analysis Agent ──MCP──► Calculator Server
              │
              └──A2A──► Writing Agent  ──MCP──► Formatter Server
```

MCP gives each agent access to **tools**. A2A lets agents access each other's **cognitive abilities**. Together, they create a modular, extensible agent ecosystem.

## 12. Validation: End-to-End A2A Protocol Tests

Let's verify every component of our A2A implementation works correctly.

In [ ]:
# ============================================================
# End-to-end validation of A2A protocol components
# ============================================================

def run_a2a_tests():
    """Comprehensive test of A2A protocol components."""
    results = []

    # Test 1: Agent Card creation and serialization
    card = AgentCard(name="TestAgent", description="A test agent",
                     capabilities=["testing", "validation"])
    d = card.to_dict()
    assert d["name"] == "TestAgent"
    assert "testing" in d["capabilities"]
    results.append(("Agent Card creation", "PASS"))

    # Test 2: Capability matching
    score = card.matches_capability("testing and validation")
    assert score > 0
    no_match = card.matches_capability("cooking recipes")
    assert no_match < score
    results.append(("Capability matching", "PASS"))

    # Test 3: Directory registration
    test_dir = AgentDirectory()
    test_dir.register(card)
    assert len(test_dir.list_all()) == 1
    assert test_dir.get("TestAgent") is not None
    results.append(("Directory registration", "PASS"))

    # Test 4: Directory discovery
    card2 = AgentCard(name="CookAgent", description="Cooks food",
                      capabilities=["cooking", "recipes", "food"])
    test_dir.register(card2)
    matches = test_dir.discover("testing")
    assert len(matches) > 0
    assert matches[0][0].name == "TestAgent"
    results.append(("Directory discovery", "PASS"))

    # Test 5: Directory unregister
    test_dir.unregister("CookAgent")
    assert test_dir.get("CookAgent") is None
    assert len(test_dir.list_all()) == 1
    results.append(("Directory unregister", "PASS"))

    # Test 6: Task Request creation
    task = TaskRequest(
        task_id="test-001", sender="Agent1",
        description="Do something", input_data={"key": "value"},
        priority=3
    )
    td = task.to_dict()
    assert td["taskId"] == "test-001"
    assert td["priority"] == 3
    results.append(("Task Request creation", "PASS"))

    # Test 7: Task Response creation
    resp = TaskResponse(
        task_id="test-001", status=TaskStatus.COMPLETED,
        result="Done!", processing_time=1.5
    )
    rd = resp.to_dict()
    assert rd["status"] == "completed"
    assert rd["processingTime"] == 1.5
    results.append(("Task Response creation", "PASS"))

    # Test 8: Task status lifecycle
    record = TaskRecord(request=task)
    assert record.status == TaskStatus.SUBMITTED
    record.update_status(TaskStatus.IN_PROGRESS)
    assert record.status == TaskStatus.IN_PROGRESS
    record.update_status(TaskStatus.COMPLETED)
    assert record.status == TaskStatus.COMPLETED
    results.append(("Task status lifecycle", "PASS"))

    # Test 9: Agent stats tracking
    stats = research_agent.get_stats()
    assert stats["name"] == "ResearchAgent"
    assert "tasks_received" in stats
    results.append(("Agent stats tracking", "PASS"))

    # Test 10: Multi-agent directory
    assert len(directory.list_all()) >= 4  # research, analysis, writing, translation, coordinator
    all_names = [c.name for c in directory.list_all()]
    assert "ResearchAgent" in all_names
    assert "AnalysisAgent" in all_names
    results.append(("Multi-agent directory", "PASS"))

    return results

tests = run_a2a_tests()
print("=== A2A Protocol Test Results ===\n")
for name, status in tests:
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon} {name}: {status}")
print(f"\n{len(tests)}/{len(tests)} tests passed ✓")

## 13. Monitoring: Agent Activity Dashboard

In a production A2A system, you'd monitor agent activity — how many tasks each agent processed, success rates, delegation patterns.

In [ ]:
# ============================================================
# Agent activity monitoring
# ============================================================

print("=== Agent Activity Dashboard ===\n")
print(f"{'Agent':<22} {'Received':>10} {'Sent':>10} {'Completed':>10} {'Failed':>8}")
print("-" * 62)

for name, agent in _agent_registry.items():
    stats = agent.get_stats()
    print(f"{stats['name']:<22} {stats['tasks_received']:>10} {stats['tasks_sent']:>10} "
          f"{stats['completed']:>10} {stats['failed']:>8}")

print()
print("=== Delegation Graph ===")
print("(Who sent tasks to whom)\n")

for name, agent in _agent_registry.items():
    if agent.task_outbox:
        for record in agent.task_outbox:
            target = "unknown"
            # Find who received it
            for other_name, other_agent in _agent_registry.items():
                for inbox_record in other_agent.task_inbox:
                    if inbox_record.request.task_id == record.request.task_id:
                        target = other_name
                        break
            print(f"  {name} ──► {target}: {record.request.description[:50]}...")

## 14. Limitations and Future Directions

### Limitations of Our Implementation
1. **No real networking**: Agents communicate via direct method calls, not HTTP. A production A2A system would use REST endpoints at `/.well-known/agent.json`.
2. **Simple capability matching**: We use keyword overlap. Production systems would use embedding-based semantic matching.
3. **No authentication/authorization**: Real agents need to verify identity before accepting tasks.
4. **No streaming**: Long-running tasks should stream progress updates. We wait for completion.
5. **No task cancellation**: Once a task starts, it runs to completion. Real A2A supports cancellation.
6. **Synchronous only**: Our agents process one task at a time. Production agents would handle concurrent tasks.

### What the Full A2A Spec Adds
- **HTTP transport** with Server-Sent Events (SSE) for streaming
- **Push notifications** for task status updates
- **Authentication** via OAuth2, API keys, or JWT
- **Multi-part artifacts** — tasks can return multiple typed outputs
- **Conversation history** — agents can have multi-turn interactions
- **Error taxonomy** with structured error codes and retry policies

## 15. Summary — Key Takeaways

### What We Built

| Component | What It Does |
|-----------|-------------|
| **AgentCard** | Capability advertisement — agents describe what they can do |
| **AgentDirectory** | Discovery service — agents find each other by capability |
| **TaskRequest / TaskResponse** | Structured work protocol with status tracking |
| **A2AAgent** | Agent that can discover, delegate to, and receive tasks from other agents |
| **CoordinatorAgent** | Meta-agent that orchestrates multi-agent pipelines |

### The Core Insight

> **A2A makes agents composable.** Just as MCP lets any agent use any tool server, A2A lets any agent collaborate with any other agent. New specialists join the ecosystem by registering — existing agents discover and use them automatically.

### The Two-Protocol Stack

```
┌───────────────────────────────────────┐
│         Agent Collaboration            │
│  (A2A: discover, delegate, compose)   │
├───────────────────────────────────────┤
│         Tool Access                    │
│  (MCP: connect, discover, call)       │
├───────────────────────────────────────┤
│         LLM Reasoning                  │
│  (Generate, plan, reflect, decide)    │
└───────────────────────────────────────┘
```

### Key Differences to Remember

| | MCP | A2A |
|---|---|---|
| **Purpose** | Agent uses tools | Agent collaborates with agents |
| **Discovery** | "What tools exist?" | "What agents can help?" |
| **Interaction** | Execute function, get result | Send task, get deliverable |
| **Architecture** | Client-server | Peer-to-peer via directory |

### Next Steps
- **Try the official A2A SDK**: Google's reference implementations in Python and TypeScript
- **Build a real agent network**: Deploy agents as HTTP services with `/.well-known/agent.json`
- **Combine MCP + A2A**: Give each agent MCP tool access AND A2A collaboration ability
- **Add semantic discovery**: Replace keyword matching with embedding-based capability search